# 03 — Generic paired metric power analysis

Use this notebook when each test instance has a score for Model A and a score for Model B.

Examples:

- per-document F1
- per-document ROUGE
- per-example BERTScore
- human score averaged per item
- any metric where each item gives a paired score `(score_A, score_B)`

The planned significance test here is a **paired t-test**. The notebook also includes a simple paired bootstrap-style p-value for teaching intuition.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from power_utils import *


## Option A: Load pilot data

The cleanest way to make assumptions is to use a pilot or dev set with paired per-instance scores.

Your CSV should have columns:

```text
item_id, model_a_score, model_b_score
```

Edit the path below to use your own file.


In [ ]:
csv_path = DATA_DIR / "demo" / "paired_metric_scores.csv"  # replace this with your own CSV path
scores_df = pd.read_csv(csv_path)
scores_df.head()

In [ ]:
pilot = estimate_inputs_from_pilot(scores_df["model_a_score"], scores_df["model_b_score"])
pd.Series(pilot).to_frame("estimated_from_pilot")

## Current paired significance test on the pilot data

This answers: is the observed difference significant on this pilot data?

Do **not** confuse this with power. Power is about how often this design would detect an assumed true effect across repeated possible datasets.


In [ ]:
from scipy.stats import ttest_rel

a = scores_df["model_a_score"].to_numpy()
b = scores_df["model_b_score"].to_numpy()

result = ttest_rel(b, a)
boot_p = paired_bootstrap_p_value(a, b, n_bootstrap=2000)

print(f"Mean A: {a.mean():.4f}")
print(f"Mean B: {b.mean():.4f}")
print(f"Observed difference B-A: {(b-a).mean():.4f}")
print(f"Paired t-test p-value: {result.pvalue:.4f}")
print(f"Paired bootstrap-style p-value: {boot_p:.4f}")

## Option B: Enter assumptions for a future study

Use the pilot estimates above as guidance, or enter your own assumptions.


In [ ]:
n_instances = ask_int("Planned number of test instances", int(pilot["n"]), minimum=20)
mean_a = ask_float("Expected mean score of Model A", round(pilot["mean_a"], 3), minimum=0.0, maximum=1.0)
expected_delta = ask_float("Expected improvement of Model B over A", round(max(pilot["delta"], 0.01), 3), minimum=0.0, maximum=1.0)
sd_a = ask_float("Expected per-instance SD for Model A", round(pilot["sd_a"], 3), minimum=0.0001)
sd_b = ask_float("Expected per-instance SD for Model B", round(pilot["sd_b"], 3), minimum=0.0001)
correlation = ask_float("Expected correlation between A and B scores", round(pilot["correlation"], 3), minimum=-0.99, maximum=0.99)
lower = ask_float("Minimum possible metric value", 0.0)
upper = ask_float("Maximum possible metric value", 1.0)
alpha = ask_float("Significance threshold alpha", 0.05, minimum=0.0001, maximum=0.5)
target_power = ask_float("Target power", 0.80, minimum=0.01, maximum=0.99)
n_trials = ask_int("Number of simulation trials", 2000, minimum=100)

mean_b = mean_a + expected_delta
params = dict(n_instances=n_instances, mean_a=mean_a, mean_b=mean_b, sd_a=sd_a, sd_b=sd_b,
              correlation=correlation, lower=lower, upper=upper, alpha=alpha,
              target_power=target_power, n_trials=n_trials)
params

## Estimate power and required n

In [ ]:
power = estimate_paired_ttest_power(
    n_instances=n_instances,
    mean_a=mean_a,
    mean_b=mean_b,
    sd_a=sd_a,
    sd_b=sd_b,
    correlation=correlation,
    lower=lower,
    upper=upper,
    alpha=alpha,
    n_trials=n_trials,
)

required_n, achieved_power = find_required_n_paired_metric(
    mean_a=mean_a,
    mean_b=mean_b,
    sd_a=sd_a,
    sd_b=sd_b,
    correlation=correlation,
    lower=lower,
    upper=upper,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
)

print(f"Estimated power at n={n_instances}: {power:.3f}")
if required_n is not None:
    print(f"Required n for {target_power:.0%} power: {required_n}")
else:
    print("Required n was not found in the search range.")

## Fixed n → minimum detectable effect

In [ ]:
mde, mde_power = find_mde_paired_metric(
    n_instances=n_instances,
    mean_a=mean_a,
    sd_a=sd_a,
    sd_b=sd_b,
    correlation=correlation,
    lower=lower,
    upper=upper,
    alpha=alpha,
    target_power=target_power,
    n_trials=n_trials,
    delta_step=0.002,
)

if mde is not None:
    print(f"Minimum detectable improvement for n={n_instances}: {mde:.3f}")
    print(f"Power at MDE: {mde_power:.3f}")
else:
    print("No MDE found within the search range.")

## Plot power as a function of the expected effect

In [ ]:
effects = np.linspace(0.005, min(0.20, upper - mean_a), 20)
rows = []
for delta in effects:
    rows.append({
        "expected_delta": delta,
        "power": estimate_paired_ttest_power(
            n_instances=n_instances,
            mean_a=mean_a,
            mean_b=mean_a + delta,
            sd_a=sd_a,
            sd_b=sd_b,
            correlation=correlation,
            lower=lower,
            upper=upper,
            alpha=alpha,
            n_trials=max(500, n_trials // 2),
        )
    })
curve = pd.DataFrame(rows)
curve.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(curve["expected_delta"], curve["power"], marker="o")
plt.axhline(target_power, linestyle="--")
plt.xlabel("Expected improvement")
plt.ylabel("Estimated power")
plt.title("Power depends on effect size and variability")
plt.show()